# Pairwise Feature Regression 02 — every exact five-minute UTC clock slot

**Primary experiment:** regress features against one another **288 times per asset**, once at **every** five-minute UTC clock position: 00:00, 00:05, 00:10, ..., 11:00, 11:05, ..., 23:55. For 11:05, we combine **only 11:05 observations across different UTC dates**; not 11:00 or 11:10. Every other five-minute slot receives the same treatment.

Within each asset × clock slot, estimate \(Y_{i}=\alpha_{ij}+\beta_{ij}X_j+\varepsilon_{ij}\) for every ordered feature pair on its own jointly available dates. Rows = response feature; columns = predictor. These are **contemporaneous in-sample** relationships, not return forecasts.

**All 288 matrices per asset are saved** in a single compressed \`02_{ASSET}_clock_matrices.npz\` file. Each archive contains seven indexable arrays: feature names, exact \`HH:MM\` labels, 288 raw-beta matrices, 288 raw-RMSE matrices, 288 intercept matrices, 288 correlations, 288 relative-RMSE matrices and 288 joint sample-count matrices. Use \`load_clock_matrix(path, "HH:MM", "beta_raw")\` or \`"error_rmse"\` to view any time; the displayed example **does not limit the computation**. Also export 288-row clock coverage reports and 288-row relationship traces.

**Small-sample caution:** v1 has only about 19 UTC dates per exact clock slot. Set \`MIN_PAIR_ROWS=6\` to expose relationships when at least six dates overlap, but label any fit with **fewer than 12 paired dates exploratory/low confidence**. Neither threshold makes slopes statistically reliable. Recent L2 instability features may appear on only ~6–7 days. Blank cells are insufficient sample, no variation or missing eligibility, never zero betas.

All v1 source transformations and scope gates are preserved. This module does not change collector data, the feature schema or any frozen confirmation model. No hourly pooling and no mixing UTC with Madrid local clock time.

## 1. Setup, parameters and in-kernel synthetic checks

In [ ]:
from pathlib import Path
import importlib.util
import json
import sys
import unittest
import gc
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
from matplotlib.colors import SymLogNorm
from dotenv import load_dotenv

ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "src" / "data_engineering").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Launch from repository root or a child folder")
sys.path.insert(0, str(ROOT / "src"))
load_dotenv(ROOT / ".env")
from data_engineering.catalog.database import RESEARCH_DATABASE
from data_engineering.modeling.full_information import FEATURE_SPECS
from data_engineering.modeling.pairwise_feature_regression import (
    model_feature_frame, EXTRAS
)
from data_engineering.modeling.pairwise_feature_clock import (
    PAIRWISE_CLOCK_VERSION, MIN_CLOCK_PAIR_ROWS, CAUTION_N_BELOW,
    fit_clock_matrices, load_clock_matrix, pair_clock_trajectory,
    save_clock_matrices, slot_labels
)

OUT = ROOT / "notebooks" / "Pairwise_Feature_Regression" / "02"
OUT.mkdir(parents=True, exist_ok=True)
ASSETS = None                  # None: every asset; or ["BTC", "ETH"]
START_DATE = None              # e.g. "2026-10-01"; None: no limit
END_DATE = None                # e.g. "2026-10-08" inclusive; None: no limit
MIN_PAIR_ROWS = MIN_CLOCK_PAIR_ROWS      # 6: exploratory, needs >=6 paired DATES
CAUTION_N = CAUTION_N_BELOW              # 12: flag any fit with <12 paired DATES
DISPLAY_ASSET = "BTC"          # ONLY for plots/display; doesn't restrict regressions
DISPLAY_CLOCK_UTC = "00:00"    # ONLY for plots/display; can choose ANY of 288 slots
EXPORT_PREVIEW_PNG = True
PAIR_TRACE_SPECS = [
    ("buy_share", "tfi"),
    ("log_atm_iv", "log_rv_1h"),
    ("obi_20_std", "obi_range"),
    ("perp_ret_5m", "return_5m"),
]
print("Version:", PAIRWISE_CLOCK_VERSION)
print("Exact UTC positions:", len(slot_labels()))
print("Minimum pair dates:", MIN_PAIR_ROWS, "| Caution below:", CAUTION_N)
print("Local DuckDB:", RESEARCH_DATABASE)
if not RESEARCH_DATABASE.is_file():
    raise FileNotFoundError(RESEARCH_DATABASE)

TEST_PATH = ROOT / "tests" / "test_pairwise_feature_clock.py"
spec = importlib.util.spec_from_file_location("_pfr_clock_tests", TEST_PATH)
if spec is None or spec.loader is None:
    raise RuntimeError("Unable to load test file: " + str(TEST_PATH))
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
suite = unittest.defaultTestLoader.loadTestsFromModule(module)
test_result = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite)
if not test_result.wasSuccessful():
    raise AssertionError("Clock-slot matrix synthetic tests failed")
print("Synthetic clock grouping tests passed in existing Jupyter kernel.")

## 2. Load same-bar feature states read-only with every relevant source column

In [ ]:
def quoted(col):
    return '"' + col.replace('"', '""') + '"'

keys = ["underlying", "state_time"]
clock_inputs = ["utc_minute_of_day", "session_progress_fraction",
                "settlement_locked_fraction", "is_weekend", "minutes_to_expiry"]
readiness = ["in_settlement_window", "perp_integrity_ok",
             "microstructure_features_ready", "instability_features_ready",
             "option_atm_quote_ok", "option_shape_quote_ok", "fit_ok",
             "option_snapshot_ready"]
source_cols = list(dict.fromkeys(
    keys + clock_inputs + readiness +
    [s.column for s in FEATURE_SPECS] + [row[1] for row in EXTRAS]
))
with duckdb.connect(str(RESEARCH_DATABASE), read_only=True) as con:
    con.execute("SET TimeZone='UTC'")
    schema = set(con.execute("DESCRIBE SELECT * FROM time_series_feature_state").df().column_name)
    if not all(k in schema for k in keys):
        raise ValueError(f"Missing state keys: {set(keys)-schema}")
    cols = [c for c in source_cols if c in schema]
    missing_cols = sorted(set(source_cols)-schema)
    sql = "SELECT " + ", ".join(map(quoted, cols)) +           " FROM time_series_feature_state WHERE 1=1"
    args = []
    if START_DATE:
        sql += " AND state_time >= CAST(? AS TIMESTAMPTZ)"
        args.append(pd.Timestamp(START_DATE, tz="UTC").isoformat())
    if END_DATE:
        sql += " AND state_time < CAST(? AS TIMESTAMPTZ)"
        args.append((pd.Timestamp(END_DATE, tz="UTC") + pd.Timedelta(days=1)).isoformat())
    if ASSETS is not None:
        if not ASSETS:
            raise ValueError("ASSETS must contain an asset or be None")
        sql += " AND underlying IN (" + ", ".join(["?"]*len(ASSETS)) + ")"
        args.extend(ASSETS)
    sql += " ORDER BY underlying, state_time"
    data = con.execute(sql, args).df()
if data.empty:
    raise ValueError("No five-minute rows for selected filters")
data["state_time"] = pd.to_datetime(data.state_time, utc=True)
if data.duplicated(["underlying", "state_time"]).any():
    raise ValueError("Duplicate asset and UTC state_time")
print("Rows:", len(data), "| assets:", sorted(data.underlying.unique()))
print("Date range:", data.state_time.min(), "-", data.state_time.max())
print("Catalog source columns missing:", missing_cols)
display(data.groupby("underlying").agg(n=("state_time","size"), first=("state_time","min"),
                                        last=("state_time","max")))

## 3. Generate 288 beta and 288 error matrices **for every asset**

Full 3D arrays are saved in compressed NumPy archives rather than producing thousands of nearly identical large CSVs. Each asset's \`02_*_clock_matrices.npz\` contains all 288 \(\beta\), RMSE, intercept, correlation, relative RMSE, and pair-count matrices. No clock position is excluded. The program processes one asset at a time and releases its arrays to protect Jupyter kernel memory.

Each \`02_{ASSET}_clock_coverage.csv\` has exactly **288 rows**, including time slots with zero valid fits, and flags pairs based on their own paired-date counts. \`02_{ASSET}_example_pair_traces.csv\` shows selected illustrative pairs across **all 288** positions; all other pairs remain stored in the full matrices.

In [ ]:
all_summaries = []
inventories = []
for asset, part in data.groupby("underlying", sort=True):
    part = part.reset_index(drop=True)
    X, inventory = model_feature_frame(part)
    inventory.insert(0, "underlying", asset)
    inventories.append(inventory)
    fitted = fit_clock_matrices(X, part.state_time, min_pair_rows=MIN_PAIR_ROWS,
                                 caution_below=CAUTION_N)
    archive = OUT / f"02_{asset}_clock_matrices.npz"
    save_clock_matrices(fitted, archive)
    summary = fitted.summary.copy()
    summary.insert(0, "underlying", asset)
    summary.to_csv(OUT / f"02_{asset}_clock_coverage.csv", index=False)
    all_summaries.append(summary)

    traces = []
    for response, predictor in PAIR_TRACE_SPECS:
        if response in fitted.features and predictor in fitted.features:
            traces.append(pair_clock_trajectory(fitted, response, predictor))
    if traces:
        pd.concat(traces, ignore_index=True).to_csv(
            OUT / f"02_{asset}_example_pair_traces.csv", index=False
        )
    print(asset, "| 5m clock slots:", len(summary),
          "| paired fits:", int(summary.n_fitted_directed_pairs.sum()),
          "| below caution N:", int(summary.n_fitted_below_caution_n.sum()),
          "| max dates per slot:", int(summary.n_dates.max()),
          "| archive:", archive.name)
    del X, inventory, fitted, traces
    gc.collect()

clock_summary = pd.concat(all_summaries, ignore_index=True)
inventory_summary = pd.concat(inventories, ignore_index=True)
clock_summary.to_csv(OUT / "02_all_clock_coverage.csv", index=False)
inventory_summary.to_csv(OUT / "02_feature_inventory.csv", index=False)
display(clock_summary.groupby("underlying").agg(
    positions=("clock_slot","nunique"),
    min_dates=("n_dates","min"),
    max_dates=("n_dates","max"),
    median_dates=("n_dates","median"),
    fitted_pairs=("n_fitted_directed_pairs","sum"),
    low_sample_fits=("n_fitted_below_caution_n","sum"),
))
assert clock_summary.groupby("underlying").size().eq(288).all()
print("288 exact UTC clock groups processed for every asset.")

## 4. View ANY of the 288 beta/error matrices (display selection does not change stored results)

In [ ]:
asset_view = DISPLAY_ASSET if DISPLAY_ASSET in set(clock_summary.underlying)     else sorted(clock_summary.underlying.unique())[0]
archive = OUT / f"02_{asset_view}_clock_matrices.npz"
beta_view = load_clock_matrix(archive, DISPLAY_CLOCK_UTC, "beta_raw")
rmse_view = load_clock_matrix(archive, DISPLAY_CLOCK_UTC, "error_rmse")
n_view = load_clock_matrix(archive, DISPLAY_CLOCK_UTC, "pair_counts")
correlation_view = load_clock_matrix(archive, DISPLAY_CLOCK_UTC, "correlation")
print("Example matrices from", asset_view, "at exact", DISPLAY_CLOCK_UTC, "UTC.")
print("The archive still includes every other five-minute slot, 00:00 through 23:55.")
print("Pair count max:", int(n_view.to_numpy().max()))
display(beta_view.iloc[:12,:12])
display(rmse_view.iloc[:12,:12])
display(n_view.iloc[:12,:12])

def preview_heatmap(mat, label, name):
    z = mat.to_numpy(dtype=float).copy()
    np.fill_diagonal(z, np.nan)
    if name == "beta_raw":
        positive = np.abs(z[np.isfinite(z)])
        cap = max(float(np.percentile(positive, 97)), 1e-12) if positive.size else 1.0
        norm = SymLogNorm(linthresh=max(cap/100, 1e-12), vmin=-cap, vmax=cap)
        values, cmap = np.clip(z, -cap, cap), "coolwarm"
    elif name == "error_rmse":
        values, norm, cmap = np.log1p(np.maximum(z, 0)), None, "viridis"
    else:
        values, norm, cmap = z, None, "coolwarm"
    fig, ax = plt.subplots(figsize=(16, 14))
    im = ax.imshow(values, interpolation="nearest", aspect="auto",
                   cmap=cmap, norm=norm, 
                   **({"vmin":-1,"vmax":1} if name == "correlation" else {}))
    labels = list(mat.index)
    ax.set_xticks(range(len(labels)), labels=labels, rotation=90, fontsize=5)
    ax.set_yticks(range(len(labels)), labels=labels, fontsize=5)
    ax.set_xlabel("Column = predictor")
    ax.set_ylabel("Row = response")
    ax.set_title(f"{asset_view} {DISPLAY_CLOCK_UTC} UTC: {label}")
    fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    fig.tight_layout()
    if EXPORT_PREVIEW_PNG:
        filename = (OUT / f"02_{asset_view}_{DISPLAY_CLOCK_UTC.replace(':','')}_{name}_preview.png")
        fig.savefig(filename, dpi=120, bbox_inches="tight")
        print("Preview:", filename.name)
    plt.show()
    plt.close(fig)

preview_heatmap(beta_view, "raw OLS beta", "beta_raw")
preview_heatmap(rmse_view, "in-sample residual RMSE", "error_rmse")
preview_heatmap(correlation_view, "standardized beta (Pearson r)", "correlation")

## 5. Check how relationships vary across all 288 UTC slots (examples, not feature selection)

In [ ]:
example = pd.read_csv(OUT / f"02_{asset_view}_example_pair_traces.csv")
fig, axs = plt.subplots(2, 2, figsize=(16, 8), sharex=True)
for ax, ((response, predictor), frame) in zip(
        axs.ravel(), example.groupby(["response", "predictor"], sort=True)):
    frame = frame.sort_values("clock_utc").reset_index(drop=True)
    x = np.arange(len(frame))
    enough = frame.n_pairs >= CAUTION_N
    ax.plot(x, frame.correlation, lw=1, alpha=0.45, label="exploratory r")
    ax.scatter(x[enough], frame.loc[enough, "correlation"], s=6,
               label=f"r, at least {CAUTION_N} paired dates")
    ax.axhline(0, linewidth=0.6)
    ax.set_title(f"{response} vs {predictor}")
    ax.set_ylim(-1.06, 1.06)
    ax.set_ylabel("same-clock Pearson r")
    ax.grid(alpha=0.2)
    ax.set_xticks([0, 48, 96, 144, 192, 240, 287],
                  labels=["00:00", "04:00", "08:00", "12:00",
                          "16:00", "20:00", "23:55"])
    ax.legend(fontsize=7)
fig.suptitle(f"{asset_view}: relationships across all 288 five-minute UTC positions")
fig.tight_layout()
if EXPORT_PREVIEW_PNG:
    fig.savefig(OUT / f"02_{asset_view}_correlation_by_clock.png",
                dpi=140, bbox_inches="tight")
plt.show()
plt.close(fig)

## 6. Write a reproducibility manifest

In [ ]:
manifest = {
    "version": PAIRWISE_CLOCK_VERSION,
    "research_question": "Per-feature ordered OLS at every exact 5m UTC clock slot across dates",
    "database_relation": "time_series_feature_state",
    "database_read_only": True,
    "current_bar_only": True,
    "no_hourly_pooling": True,
    "n_clock_slots_per_asset": 288,
    "clock_label_timezone": "UTC",
    "clock_positions": slot_labels().tolist(),
    "min_pair_observations": MIN_PAIR_ROWS,
    "low_sample_caution_if_below_n": CAUTION_N,
    "start_date_filter": START_DATE,
    "end_date_filter": END_DATE,
    "asset_filter": ASSETS,
    "processed_assets": sorted(clock_summary.underlying.unique().tolist()),
    "input_rows": len(data),
    "first_state_utc": data.state_time.min().isoformat(),
    "last_state_utc": data.state_time.max().isoformat(),
    "n_feature_specs": len(FEATURE_SPECS),
    "missing_source_columns": missing_cols,
    "primary_artifact": "02_{ASSET}_clock_matrices.npz",
    "array_axes": "clock_utc_slot(288), response_feature, predictor_feature",
    "array_keys": ["features", "clock_utc", "beta_raw", "error_rmse",
                   "intercept", "correlation", "relative_rmse", "pair_counts"],
    "display_clock_is_example_only": DISPLAY_CLOCK_UTC,
    "grouping": "same UTC time across different UTC calendar dates, exact 5m position",
    "no_global_complete_case": True,
    "no_imputation": True,
    "no_test_data_claim": True,
    "interpretation": "exploratory in-sample associations with <=~19 dates per slot",
    "created_utc": pd.Timestamp.now(tz="UTC").isoformat(),
}
(OUT / "02_run_metadata.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8")
print("Every 5-minute clock position generated, including unused/NaN cells.")
print("Commit executed notebook and the complete 02/ directory for review.")
print("To inspect any exact time: load_clock_matrix(archive, '13:25', 'error_rmse')")

### Scientific interpretation

A clock-conditioned same-bar regression is **not a causally valid forecast** or a trading signal. With roughly 19 dates per slot, a seven-date L2 relationship and a 12-date options relationship can easily look extreme by chance. Treat fits with \(n<12\) as low-sample exploratory; even \(n\ge12\) remains too small for strong inference. Correlation varies with markets, quote eligibility, source revisions and expiry structure; it is not evidence of daily repeatability.

**All 288 positions are analyzed, whether or not the notebook displays their PNG.** To retrieve a specific slot, change \`DISPLAY_CLOCK_UTC\` and rerun only the visualization cell; there is no need to recompute the regression or discard other slots.